In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import pandas as pd
import os
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import numpy as np

# Load the CSV file containing image labels
train_df = pd.read_csv('data/archive/Train.csv')

# Correct image paths by removing extra 'Train' folder if necessary
train_df['Path'] = train_df['Path'].str.replace('Train/Train', 'Train', regex=False)

# Prepare image data and labels
images = []
labels = []

for index, row in train_df.iterrows():
    img_path = os.path.join('data/archive/Train', row['Path'])
    if os.path.exists(img_path):
        image = load_img(img_path, target_size=(32, 32))
        image = img_to_array(image) / 255.0
        images.append(image)
        labels.append(row['ClassId'])
    else:
        print(f"Image not found: {img_path}")

X = np.array(images)
y = to_categorical(labels, num_classes=len(train_df['ClassId'].unique()))

# Define CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(train_df['ClassId'].unique()), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X, y, epochs=25, batch_size=32)

# Save the trained model
os.makedirs('models', exist_ok=True)
model.save('models/traffic_sign_model.h5')

print("Training completed using images from the train folder and labels from Train.csv, and model saved successfully!")